# Taller de Deep Learning - version local Windows

Este notebook asume que estas en la carpeta del proyecto y que ya existen:

- `.venv` con Python 3.11
- `./TTS` clonado e instalado
- `./data/LJSpeech-1.1/metadata_subset.csv`

Si aun no has descargado el dataset, ejecuta primero `download_ljspeech_subset.ps1`.


In [1]:
import sys
import torch

print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


C:\Users\USER\Documents\Yamin\.venv\Scripts\python.exe
2.11.0+cu128
12.8
True
1
NVIDIA GeForce RTX 5060


In [2]:
from pathlib import Path
import json
import os
import subprocess

import torch

ROOT = Path.cwd()
PYTHON_EXE = ROOT / ".venv" / "Scripts" / "python.exe"
TTS_DIR = ROOT / "TTS"
DATASET_DIR = ROOT / "data" / "LJSpeech-1.1"
CHECKPOINTS_DIR = ROOT / "checkpoints"
CONFIG_PATH = ROOT / "config.local.json"
RESULT_WAV = ROOT / "resultado_taller.wav"

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"Python listo: {PYTHON_EXE.exists()}")
print(f"TTS listo: {TTS_DIR.exists()}")
print(f"Dataset listo: {(DATASET_DIR / 'metadata_subset.csv').exists()}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


ROOT: C:\Users\USER\Documents\Yamin
Python listo: True
TTS listo: True
Dataset listo: True
CUDA disponible: True
GPU: NVIDIA GeForce RTX 5060


In [3]:
config = {
    "run_name": "tacotron_taller_3",
    "run_description": "Tacotron2 local Windows",
    "epochs": 20,
    "batch_size": 8,
    "eval_batch_size": 4,
    "mixed_precision": torch.cuda.is_available(),
    "num_loader_workers": 0,
    "num_eval_loader_workers": 0,
    "run_eval": True,
    "test_delay_epochs": -1,
    "print_step": 10,
    "print_eval": False,
    "save_step": 50,
    "save_checkpoint": True,
    "save_n_checkpoints": 3,
    "save_best_after": 0,
    "target_loss": "loss",
    "cudnn_benchmark": False,
    "clear_opt_state": False,
    "output_path": CHECKPOINTS_DIR.as_posix(),
    "datasets": [
        {
            "formatter": "ljspeech",
            "meta_file_train": "metadata_subset.csv",
            "path": DATASET_DIR.as_posix()
        }
    ],
    "model": "tacotron",
    "model_args": {
        "num_chars": 100,
        "embedding_dim": 512,
        "prenet_dim": 256,
        "postnet_num_layers": 5,
        "decoder_dim": 1024
    },
    "audio": {
        "sample_rate": 22050,
        "num_mels": 80,
        "min_level_db": -100,
        "frame_shift_ms": 12.5,
        "frame_length_ms": 50,
        "ref_level_db": 20,
        "power": 1.5,
        "griffin_lim_iters": 60,
        "signal_norm": True,
        "symmetric_norm": True,
        "max_norm": 4.0,
        "clip_norm": True,
        "mel_fmin": 0.0,
        "mel_fmax": 8000.0,
        "do_trim_silence": True
    }
}

with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print(f"Archivo generado: {CONFIG_PATH}")


Archivo generado: C:\Users\USER\Documents\Yamin\config.local.json


In [4]:
required_paths = [
    PYTHON_EXE,
    TTS_DIR / "TTS" / "bin" / "train_tts.py",
    DATASET_DIR / "metadata_subset.csv",
    CONFIG_PATH,
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Faltan rutas necesarias:\n" + "\n".join(missing))

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

cmd = [
    str(PYTHON_EXE),
    str(TTS_DIR / "TTS" / "bin" / "train_tts.py"),
    "--config_path",
    str(CONFIG_PATH),
]

print("Ejecutando entrenamiento local...")
subprocess.run(cmd, check=True, env=env, cwd=str(ROOT))


Ejecutando entrenamiento local...


CompletedProcess(args=['C:\\Users\\USER\\Documents\\Yamin\\.venv\\Scripts\\python.exe', 'C:\\Users\\USER\\Documents\\Yamin\\TTS\\TTS\\bin\\train_tts.py', '--config_path', 'C:\\Users\\USER\\Documents\\Yamin\\config.local.json'], returncode=0)

In [5]:
import glob

run_folders = glob.glob(str(CHECKPOINTS_DIR / "tacotron_taller_3*"))
if not run_folders:
    print("Todavia no hay carpetas de entrenamiento en checkpoints.")
else:
    for folder in sorted(run_folders):
        print(folder)
        for ckpt in sorted(Path(folder).glob("checkpoint_*.pth"))[-5:]:
            size_mb = ckpt.stat().st_size / (1024 * 1024)
            print(f"  {ckpt.name} - {size_mb:.1f} MB")


C:\Users\USER\Documents\Yamin\checkpoints\tacotron_taller_3-March-27-2026_04+34PM-0000000
C:\Users\USER\Documents\Yamin\checkpoints\tacotron_taller_3-March-27-2026_04+51PM-0000000
C:\Users\USER\Documents\Yamin\checkpoints\tacotron_taller_3-March-27-2026_04+57PM-0000000
  checkpoint_350.pth - 77.9 MB
  checkpoint_400.pth - 77.9 MB
  checkpoint_450.pth - 77.9 MB


In [9]:
import glob
import json
import os
from IPython.display import Audio, display
from TTS.utils.manage import ModelManager
from TTS.utils.synthesizer import Synthesizer

USE_PRETRAINED_VOCODER = True
VOCODER_NAME = "vocoder_models/en/ljspeech/hifigan_v2"

run_folders = glob.glob(str(CHECKPOINTS_DIR / "tacotron_taller_3*"))
if not run_folders:
    raise RuntimeError("No se encontro la carpeta del modelo en checkpoints.")

latest_run = max(run_folders, key=os.path.getctime)

best_model = os.path.join(latest_run, "best_model.pth")
best_model_candidates = glob.glob(os.path.join(latest_run, "best_model*.pth"))
if os.path.exists(best_model):
    model_path = best_model
elif best_model_candidates:
    model_path = max(best_model_candidates, key=os.path.getctime)
else:
    checkpoints = glob.glob(os.path.join(latest_run, "checkpoint_*.pth"))
    if not checkpoints:
        raise RuntimeError("No se encontraron archivos de modelo.")
    model_path = max(checkpoints, key=os.path.getctime)

config_path = os.path.join(latest_run, "config.json")
with open(config_path, "r", encoding="utf-8") as f:
    run_config = json.load(f)
tts_out_channels = run_config.get("out_channels")
use_cuda = torch.cuda.is_available()
vocoder_path = None
vocoder_config_path = None

if USE_PRETRAINED_VOCODER and tts_out_channels == 80:
    try:
        manager = ModelManager(progress_bar=True, verbose=True)
        vocoder_path, vocoder_config_path, _ = manager.download_model(VOCODER_NAME)
        print(f"Vocoder: {VOCODER_NAME}")
    except Exception as error:
        print(f"No se pudo cargar el vocoder. Se usara Griffin-Lim. Error: {error}")
        vocoder_path = None
        vocoder_config_path = None
elif USE_PRETRAINED_VOCODER:
    print(f"No se usara vocoder. Este modelo genera {tts_out_channels} canales y HiFiGAN espera 80 mel bins.")

print(f"Modelo: {model_path}")
print(f"CUDA: {use_cuda}")

synthesizer = Synthesizer(
    tts_checkpoint=model_path,
    tts_config_path=config_path,
    vocoder_checkpoint=vocoder_path,
    vocoder_config=vocoder_config_path,
    use_cuda=use_cuda,
)

texto_prueba = "Hello, this is a voice trained with artificial intelligence."
wav = synthesizer.tts(texto_prueba)
synthesizer.save_wav(wav, str(RESULT_WAV))

print(f"Audio generado: {RESULT_WAV}")
display(Audio(str(RESULT_WAV)))


Modelo: C:\Users\USER\Documents\Yamin\checkpoints\tacotron_taller_3-March-27-2026_04+57PM-0000000\best_model.pth
 > Using model: tacotron
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:12
 | > frame_length_ms:50
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:True
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:45
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Model's reduction rate `r` is set to: 2
 > Text splitted to sentences.
['Hello, this is a voice trained with artificial